# 2 Use the LLM Gateway

**Purpose:**
Call the model you deployed in Section 1 the governed way: through the LLM Gateway, under a profile.

**What we're doing**

We set a profile name and a gateway model name, derive the gateway's URL from the endpoint URL you already have, and send the same request we sent in Section 1, this time through the gateway.

**Why we're doing it**

* In Section 1, every client needed the raw endpoint URL and the exact Model Endpoint Name. The gateway gives clients one route for every model.
* The gateway decides who may call which model, through **profiles**, and meters what they use
* To see exactly what changes in a request when it goes through the gateway, and what stays the same

**What to look for**

* The gateway URL is the same host as your endpoint, with `/llm` as the path
* Two things a direct call does not need: the `X-ClearML-Profile` header and the gateway's own model name
* The same model answering the same question by both routes

**Output**

A working chat completion through the LLM Gateway.


## 2.1 What Changes When You Go Through the Gateway

The request is still an OpenAI-compatible chat completion with the same gateway token. Three things change:

| | Section 1: direct to the endpoint | Section 2: through the LLM Gateway |
|---|---|---|
| **URL** | `https://<host>/service/<id>/<endpoint name>/v1` | `https://<host>/llm/v1` |
| **`"model"`** | The **Model Endpoint Name** | The model's name **in your profile** |
| **Headers** | `Authorization` | `Authorization` and `X-ClearML-Profile` |

A **profile** is the LLM Gateway's unit of governance. It lists the models its members may call, and the gateway checks every request against it. A model in a profile can be one you deployed on Klique, such as your vLLM endpoint, or an external provider's model. Clients call them all the same way.

> **Note:** The gateway's model name is set when the model is added to the profile, so it is often different from the Model Endpoint Name. The gateway does not accept the endpoint name.


## 2.2 Get Your Profile and Model Name

You need two values:

| Value | Example | Where it comes from |
|---|---|---|
| **Profile name** | `frank-test` | The LLM Gateway profile you belong to |
| **Gateway model name** | `vllm-qwen-franktest` | The name your Section 1 endpoint has in that profile |

Profiles are set up in the LLM Gateway, usually by your Klique administrator. If you do not have a profile yet, ask your administrator to add your Section 1 endpoint to a profile you belong to, and to tell you the model name it has there. See [LLM Gateway](https://klique.ai/docs/latest/docs/whats_klique/llm_gateway).


## 2.3 Set Your Profile and Model Name

Neither value is a secret, so set them directly in the cell below. Replace the examples with your own.

In [ ]:
PROFILE = "frank-test"                 # Your LLM Gateway profile
GATEWAY_MODEL = "vllm-qwen-franktest"  # Your Section 1 endpoint's name in that profile

## 2.4 Load the Token and the Gateway URL

The next cell loads everything else the notebook needs, without asking you for anything:

- `sys.path.insert(0, "..")` adds the project root to the places Python looks for modules. This notebook runs from `02_LLMGateway/`, so without it, `import config` would fail.
- Importing from `config` reads the `.env` file you created in Section 1.6. The cell takes three values from it: the endpoint URL (`base_url`), the **Model Endpoint Name** (`model`), and the gateway token (`token`). The two names you set in 2.3 are not in `.env`.
- `KLIQUE_LLM_GATEWAY_URL` is not in `.env` either. `config.py` derives it from `KLIQUE_BASE_URL`: the LLM Gateway is always at `<hostname>/llm`, on the same host as your model endpoint.
- The `print` lines show what was loaded, so you can check it before calling the gateway.

In [ ]:
import sys
sys.path.insert(0, "..")
from config import KLIQUE_BASE_URL as base_url, KLIQUE_MODEL as model, KLIQUE_GATEWAY_TOKEN as token
from config import KLIQUE_LLM_GATEWAY_URL as gateway_url
import requests

print(f"Endpoint (Section 1): {base_url}")
print(f"LLM Gateway:          {gateway_url}")
print(f"Profile:              {PROFILE}")
print(f"Gateway model:        {GATEWAY_MODEL}")
print(f"Token:                {token[:8]}...")

Compare the first two lines. The gateway URL keeps the endpoint's host and replaces the whole path with `/llm`. As in Section 1, the token is truncated on purpose.


## 2.5 Call the Model Through the Gateway

The helper below sends a chat completion to the gateway. It is the Section 1 request with three changes: the gateway URL, the `X-ClearML-Profile` header, and the gateway model name.

In [ ]:
def chat(prompt, profile_name, model_name, max_tokens=256, timeout=60):
    """Send a chat completion through the LLM Gateway and return the parsed reply."""
    r = requests.post(
        f"{gateway_url}/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {token}",
            "X-ClearML-Profile": profile_name,
        },
        json={
            "model": model_name,
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": max_tokens,
        },
        timeout=timeout,
    )
    if not r.ok:
        # Show the gateway's explanation, e.g. 401 (bad token) or 404 (unknown model).
        print(f"❌ HTTP {r.status_code}: {r.text[:300]}")
    r.raise_for_status()
    return r.json()


reply = chat("In one sentence, what is AI?", PROFILE, GATEWAY_MODEL)
print("✅ Response:")
print(reply["choices"][0]["message"]["content"])
print(f"\nServed by:   {reply['model']}")
print(f"Tokens used: {reply['usage']['total_tokens']}")

Look at **Served by**. You asked for the gateway model name, but the reply names your **Model Endpoint Name** from Section 1. The gateway mapped the profile's name to your endpoint and forwarded the request to it.

> **Checkpoint:** the cell prints `✅ Response:` followed by text. Your model is answering through the LLM Gateway.

> **Tip:** If the reply starts with `<think>` and stops mid-sentence, your model is a reasoning model such as Qwen3. Raise `max_tokens`, or see `01_vLLM/vllm_klique_test_clean.ipynb` for how to separate the thinking from the answer.


## 2.6 The Gateway Checks Every Request

The profile is not a label; the gateway enforces it. Send the same request with the Section 1 **Model Endpoint Name** instead of the gateway model name:

In [ ]:
try:
    chat("In one sentence, what is Klique?", PROFILE, model)
except requests.HTTPError:
    print(f"Rejected: '{model}' is not a model in profile '{PROFILE}'.")

The gateway returns `404` with `model_not_found`. It accepts only the models your profile lists, under the names the profile gives them. It returns the same error for a profile name you do not belong to, so a wrong `PROFILE` looks the same as a wrong `GATEWAY_MODEL`.

> **Note:** Your Section 1 endpoint still answers direct calls with your gateway token. The LLM Gateway governs the traffic that goes through it; send your applications through the gateway so that access, budgets, and usage tracking apply to them.


## 2.7 Where the Usage Goes

Every call through the gateway is metered per user and per model. Open the **AI Usage** dashboard in the WebApp to see the calls you just made, with their token counts.

<!-- screenshot: images/ai_usage.png (AI Usage dashboard showing the calls from 2.5) -->


## Troubleshooting

| Symptom | Likely cause |
|---|---|
| `404` with `model_not_found` | `GATEWAY_MODEL` is not a model in `PROFILE`, or you do not belong to that profile. Check both values with your administrator. |
| `404` with `Route Not Found` | The path is wrong. Requests go to `<hostname>/llm/v1/chat/completions`. The gateway does not serve `/v1/models`. |
| `401` | The gateway token is wrong, expired, or revoked. Generate a new one as in Section 1.5. |
| `502`, `503`, or a timeout | The gateway reached the profile's model, but the model did not answer. Check that your Section 1 instance is still **Active**; idle shutdown may have stopped it. |


## Wrap-Up

You have now followed the whole path:

```
Model → vLLM Model Deployment → Model Endpoint → AI Application Gateway → LLM Gateway
```

Your application needs only the gateway URL, a profile, a model name, and a token. Which machine serves the model, and whether it is a vLLM instance on Klique or an external provider, is behind the gateway.

**Next:** In `02b_Models_and_Routed_Resources.ipynb` we call several models through the same gateway, then let a Routed Resource choose the model for us. Leave your Section 1 instance running.


## Related Docs

- [LLM Gateway](https://klique.ai/docs/latest/docs/whats_klique/llm_gateway)
- [AI Application Gateway](https://klique.ai/docs/latest/docs/whats_klique/appgw)
- [Model Endpoints](https://klique.ai/docs/latest/docs/webapp/webapp_model_endpoints)